In [1]:
# 31a_Baseline_FingerRule: baseline cổ điển (luật ngón tay) trên test cũ 48 video, cùng landmark MediaPipe, không train.
import os, glob, time, numpy as np, pandas as pd
from game_logic import hand_shape, majority_latest, CLASSES      # game_logic.py để cùng thư mục với notebook
ROOT      = "."                    # thư mục gốc chứa landmark (tìm đệ quy theo TÊN FILE nên không phụ thuộc cấu trúc thư mục)
MANIFEST  = "manifest_all.csv"
TAP       = "test"                 # test cũ (48 video); đặt None để chạy mọi tập (chỉ để thử code)
WIN_SEC   = 0.2                    # cửa sổ cuối video để đọc cử chỉ cuối (bằng confirm_sec của game, tham số tự đặt)
OUT       = "baseline_out"; os.makedirs(OUT, exist_ok=True)

m = pd.read_csv(MANIFEST); m["f"] = m.path.str.replace("\\", "/", regex=False).str.split("/").str[-1]
if TAP: m = m[m.tap == TAP]
idx = {}
for p in glob.glob(os.path.join(ROOT, "**", "*.npz"), recursive=True):
    idx[os.path.basename(p).split("_", 1)[1] if os.path.basename(p)[:1].isdigit() else os.path.basename(p)] = p   # bỏ tiền tố số nếu có
m["file"] = m.f.map(idx)
print(f"{TAP}: manifest {len(m)} video, tìm thấy file {m.file.notna().sum()}, thiếu {m.file.isna().sum()}")
m = m[m.file.notna()].reset_index(drop=True)


test: manifest 48 video, tìm thấy file 48, thiếu 0


In [2]:
# Chạy luật ngón tay từng khung (giả lập thời gian thực: gọi từng khung một), ghi kết quả từng video.
rows, ms_all = [], []
for r in m.itertuples():
    d = np.load(r.file); lm, fps = d["lm"], float(d["fps"]); n = len(lm)
    pred = np.empty(n, int); ts = []
    for i in range(n):
        t0 = time.perf_counter(); pred[i] = int(hand_shape(lm[i])); ts.append((time.perf_counter() - t0) * 1000)
    ms_all += ts
    ok = pred >= 0; k = max(1, round(WIN_SEC * fps))
    last = [int(v) for v in pred[-k:] if v >= 0]                       # cử chỉ cuối: đa số trong WIN_SEC cuối, hòa phiếu lấy cái muộn nhất (như game)
    fin = majority_latest(last)
    hit = np.where(ok & (pred == r.nhan))[0]
    rows.append(dict(file=r.f, nguon=r.nguon, lop=r.lop, nhan=r.nhan, khung=n, fps=fps, khung_co_tay=int(ok.sum()),
                     khung_dung=int((ok & (pred == r.nhan)).sum()), khung_khong_xd=int((~ok).sum()),
                     cuoi=-1 if fin is None else fin, khung_dung_dau_tien_s=round(hit[0] / fps, 3) if len(hit) else np.nan,
                     co_it_nhat_1_khung_dung=bool(len(hit))))
R = pd.DataFrame(rows); R.to_csv(f"{OUT}/finger_rule_per_video.csv", index=False)
print("ms mỗi khung (hand_shape, 1 khung/lần): TB %.3f | trung vị %.3f | p95 %.3f | max %.3f" % (np.mean(ms_all), np.median(ms_all), np.percentile(ms_all, 95), np.max(ms_all)))


ms mỗi khung (hand_shape, 1 khung/lần): TB 0.020 | trung vị 0.017 | p95 0.027 | max 0.754


In [3]:
# Tổng hợp: accuracy khung, F1 macro, ma trận nhầm lẫn (cử chỉ cuối mỗi video), tỉ lệ không xác định, theo từng nguồn.
def f1_macro(y, p):
    f = []
    for c in range(3):
        tp = np.sum((y == c) & (p == c)); fp = np.sum((y != c) & (p == c)); fn = np.sum((y == c) & (p != c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))
def conf(y, p):
    M = np.zeros((3, 4), int)                                          # hàng: nhãn thật | cột: paper, rock, scissors, không xác định
    for a, b in zip(y, p): M[a, b if b >= 0 else 3] += 1
    return pd.DataFrame(M, index=[f"thật {c}" for c in CLASSES], columns=CLASSES + ["không xác định"])
def summ(D):
    y, p = D.nhan.values, D.cuoi.values
    return dict(so_video=len(D),
                acc_moi_khung=round(100 * D.khung_dung.sum() / D.khung_co_tay.sum(), 1),
                ty_le_khung_khong_xd=round(100 * D.khung_khong_xd.sum() / D.khung.sum(), 1),
                acc_cu_chi_cuoi_video=round(100 * np.mean(y == p), 1), f1_macro_cuoi=round(100 * f1_macro(y, p), 1),
                video_khong_xd_cuoi=int((p < 0).sum()),
                video_co_it_nhat_1_khung_dung=round(100 * D.co_it_nhat_1_khung_dung.mean(), 1))
S = pd.DataFrame({"tất cả": summ(R), **{s: summ(g) for s, g in R.groupby("nguon")}}).T
S.to_csv(f"{OUT}/finger_rule_summary.csv"); print(S.to_string())
C = conf(R.nhan.values, R.cuoi.values); C.to_csv(f"{OUT}/finger_rule_confusion.csv"); print("\nMa trận nhầm lẫn (cử chỉ cuối video):\n", C.to_string())
print(f"\nGhi chú: không có ngưỡng tin cậy nên 'video có ít nhất 1 khung đúng' KHÔNG tương đương 96,7% (đúng và đủ tự tin) của TCN; không so trực tiếp hai con số này.")


                 so_video  acc_moi_khung  ty_le_khung_khong_xd  acc_cu_chi_cuoi_video  f1_macro_cuoi  video_khong_xd_cuoi  video_co_it_nhat_1_khung_dung
tất cả               48.0           82.8                   0.8                   95.8           96.8                  1.0                           97.9
Dataset_bonus        15.0           70.1                   0.3                   93.3           93.3                  0.0                           93.3
Dataset_hand         12.0           75.7                   2.6                   91.7           95.2                  1.0                          100.0
dataset_labeled      21.0           96.7                   0.0                  100.0          100.0                  0.0                          100.0

Ma trận nhầm lẫn (cử chỉ cuối video):
                paper  rock  scissors  không xác định
thật paper        16     0         0               0
thật rock          0    14         1               1
thật scissors      0     0        16